# affect_aif Demo Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/har5h1l/affect_aif/blob/master/notebooks/demo.ipynb)

This is the guided public walkthrough for the project. It runs the small demo configs, then turns each output into a mechanism-level read: where confidence moves, when policy entropy changes, and why payoff is only one part of the story.

The demos mirror the numbered paper configs but use fewer seeds, shorter episodes, and reduced profile grids. Treat the plots as an executable map of the mechanism, not as publication-scale evidence.


## Tour Map

The notebook has two layers.

First, the core paper path:

1. Predictability over value: confidence follows partner reliability more than reward alone.
2. Deployment ablation: confidence matters when it reaches policy precision.
3. Partner selection: local confidence changes who the agent samples.
4. Betrayal adaptation: accumulated confidence can help or hurt after a stance change.

Then the appendix-level extensions:

5. Alpha sweep: gain changes confidence amplitude and commitment.
6. Prior x gain factorial: starting beliefs and gain form computational profiles.
7. Forgiveness: reengagement and restored confidence can separate.

Each section asks a question, runs a current `configs/demo/` TOML, shows compact summaries, and points back to the matching paper config.


## 1. Bootstrap Runtime

In Colab, this clones the selected branch and moves into the repo. Locally, it searches upward from the notebook directory. The command-line scripts remain the source of truth; the notebook is a guided interface over those scripts.


In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys

try:
    import google.colab  # type: ignore[import-not-found]
    IN_COLAB = True
except Exception:
    IN_COLAB = Path("/content").exists()

def sanitize_repo_url(value):
    value = str(value).strip()
    if value.startswith("[") and "](" in value and value.endswith(")"):
        value = value.split("](", 1)[1][:-1]
    if value.startswith("<") and value.endswith(">"):
        value = value[1:-1]
    return value.strip()


REPO_URL = sanitize_repo_url(os.environ.get("AFFECT_AIF_REPO_URL", "https://github.com/har5h1l/affect_aif.git"))
BRANCH = os.environ.get("AFFECT_AIF_BRANCH", "master")


def clone_repo(repo_url, branch, repo_dir):
    cmd = ["git", "clone", "--depth", "1", "--branch", branch, repo_url, str(repo_dir)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("Git clone failed.")
        print("Command:", " ".join(cmd))
        print("stdout:", result.stdout.strip() or "<empty>")
        print("stderr:", result.stderr.strip() or "<empty>")
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout, stderr=result.stderr)


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts/experiment/run.py").exists() and (candidate / "configs/demo").exists():
            return candidate
    return None


existing_root = find_repo_root(Path.cwd())
if existing_root is not None:
    os.chdir(existing_root)
elif IN_COLAB:
    repo_dir = Path("/content/affect_aif")
    if repo_dir.exists() and not (repo_dir / ".git").exists():
        shutil.rmtree(repo_dir)
    if not repo_dir.exists():
        clone_repo(REPO_URL, BRANCH, repo_dir)
    os.chdir(repo_dir)
else:
    raise RuntimeError("Could not find affect_aif repo root from the current notebook directory.")

print("Working directory:", Path.cwd())
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


## 2. Install Dependencies

Colab runtimes are fresh, so install the repo there by default. Local users can set `INSTALL_DEPS = True` if their kernel does not already have the package installed.


In [ ]:
INSTALL_DEPS = IN_COLAB

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
    print("Installed affect_aif in editable mode for this runtime.")
else:
    print("Skipped dependency install. Set INSTALL_DEPS = True if imports fail.")


## 3. Check Runtime Device

These demos are designed to run on CPU. The device check is here so public readers can see what JAX is using before they launch experiments.


In [ ]:
def detect_accelerator():
    try:
        import jax
        devices = [str(device) for device in jax.devices()]
    except Exception as exc:
        return "gpu" if shutil.which("nvidia-smi") else "cpu", [f"JAX unavailable: {exc}"]

    gpu_devices = [device for device in devices if "gpu" in device.lower() or "cuda" in device.lower()]
    if gpu_devices:
        return "gpu", devices
    return "cpu", devices

DEVICE, JAX_DEVICES = detect_accelerator()
print("Selected device:", DEVICE)
print("JAX devices:", JAX_DEVICES)

if DEVICE == "gpu" and shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
elif DEVICE == "cpu":
    print("No GPU detected; the demo will run on CPU.")


## 4. Demo Controls And Analysis Helpers

All seven demo experiments are selected by default because they are meant to be runnable in one sitting. Set `RUN_DEMO = False` only if you already have outputs under `outputs/` and want to re-run the display cells.

The helper functions below do three jobs: run configs through `scripts/experiment/run.py`, run generic analysis through `scripts/analysis/analyze.py`, and display mechanism snapshots that stay robust across the different demo shapes.


In [ ]:
RUN_DEMO = True
RUN_ANALYSIS = True
RUN_OPTIONAL_DEMOS = True
RESET_OUTPUTS = False
OUTPUT_ROOT = Path("outputs")
DEMO_BATCH_PREFIX = "notebook_demo"
SELECTED_DEMOS = ["predictability_value", "deployment_ablation", "partner_selection", "betrayal_adaptation", "alpha_sweep", "prior_factorial", "forgiveness"]

DEMO_EXPERIMENTS = {
    "predictability_value": {
        "config": "configs/demo/01_predictability_value.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_01_predictability_value",
        "paper_config": "configs/paper/01_predictability_value.toml",
        "label": "1. Predictability over value",
        "question": "Does confidence track partner predictability more directly than realized payoff?",
        "readout": "Compare payoff, policy entropy, predictive evidence, and model-fitness tables.",
        "optional": False,
    },
    "deployment_ablation": {
        "config": "configs/demo/02_deployment_ablation.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_02_deployment_ablation",
        "paper_config": "configs/paper/02_deployment_ablation.toml",
        "label": "2. Deployment ablation",
        "question": "Does affective precision matter because it changes action deployment, not because it adds new observations?",
        "readout": "Look for entropy and action-deployment changes even when knowledge readouts remain close.",
        "optional": False,
    },
    "partner_selection": {
        "config": "configs/demo/03_partner_selection.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_03_partner_selection",
        "paper_config": "configs/paper/03_partner_selection.toml",
        "label": "3. Partner selection",
        "question": "Does partner-local precision reshape who the agent chooses to engage?",
        "readout": "Inspect selected-partner rates, partner-type summaries, and entropy before treating payoff as the main signal.",
        "optional": False,
    },
    "betrayal_adaptation": {
        "config": "configs/demo/04_betrayal_adaptation.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_04_betrayal_adaptation",
        "paper_config": "configs/paper/04_betrayal_adaptation.toml",
        "label": "4. Betrayal adaptation",
        "question": "What happens when confidence accumulated under cooperation meets a sudden stance change?",
        "readout": "Read entropy, joint accuracy, reallocation, and post-switch phase summaries together.",
        "optional": False,
    },
    "alpha_sweep": {
        "config": "configs/demo/05a_alpha_sweep.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_05a_alpha_sweep",
        "paper_config": "configs/paper/05a_alpha_sweep.toml",
        "label": "5a. Alpha sweep",
        "question": "How does gain change confidence amplitude and policy commitment?",
        "readout": "Use the appendix-style profile plots: gain can scale confidence without producing a monotone payoff ranking.",
        "optional": False,
    },
    "prior_factorial": {
        "config": "configs/demo/05b_prior_factorial.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_05b_prior_factorial",
        "paper_config": "configs/paper/05b_prior_factorial.toml",
        "label": "5b. Prior x gain factorial",
        "question": "How do starting trust priors and precision gain combine into computational profiles?",
        "readout": "Compare profile-level payoff, entropy, partner choice, and compact phenotype validation tables.",
        "optional": True,
    },
    "forgiveness": {
        "config": "configs/demo/05c_forgiveness.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_05c_forgiveness",
        "paper_config": "configs/paper/05c_forgiveness.toml",
        "label": "5c. Forgiveness",
        "question": "Can reengagement recover without restored confidence?",
        "readout": "Read post-switch reengagement, beta movement, and payoff recovery as separate axes.",
        "optional": True,
    },
}

for spec in DEMO_EXPERIMENTS.values():
    if not Path(spec["config"]).exists():
        raise FileNotFoundError(spec["config"])
    if not Path(spec["paper_config"]).exists():
        raise FileNotFoundError(spec["paper_config"])

if RESET_OUTPUTS:
    for spec in DEMO_EXPERIMENTS.values():
        batch_dir = OUTPUT_ROOT / spec["batch"]
        if batch_dir.exists():
            shutil.rmtree(batch_dir)

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)


def selected_demo_names():
    if SELECTED_DEMOS == "all":
        return list(DEMO_EXPERIMENTS)
    return list(SELECTED_DEMOS)


def run_required(cmd, required_path=None):
    if required_path is not None and not Path(required_path).exists():
        raise FileNotFoundError(required_path)
    print("Running:", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)


def result_paths_for_batch(batch_name):
    batch_dir = OUTPUT_ROOT / batch_name
    return sorted(batch_dir.glob("**/results.csv"))


def analysis_dirs_for_paths(paths):
    return sorted({path.parent / "notebook_analysis" for path in paths if (path.parent / "notebook_analysis").exists()})


def run_demo_experiment(name):
    spec = DEMO_EXPERIMENTS[name]
    if name not in selected_demo_names():
        print(f"Skipping {spec['label']}. Add {name!r} to SELECTED_DEMOS to run it.")
        return []
    if spec.get("optional") and not RUN_OPTIONAL_DEMOS:
        print(f"Skipping optional demo: {spec['label']}. Set RUN_OPTIONAL_DEMOS = True to run it.")
        return []

    cmd = [
        sys.executable,
        "scripts/experiment/run.py",
        "--config",
        spec["config"],
        "--output-dir",
        str(OUTPUT_ROOT),
        "--batch-name",
        spec["batch"],
        "--workers",
        "1",
    ]
    if RUN_DEMO:
        run_required(cmd)
    else:
        print("Experiment run skipped. Set RUN_DEMO = True to execute", name)

    paths = result_paths_for_batch(spec["batch"])
    if not paths:
        raise RuntimeError(f"No results.csv files found for {name} under {OUTPUT_ROOT / spec['batch']}.")
    if RUN_ANALYSIS:
        for result_path in paths:
            run_required(
                [
                    sys.executable,
                    "scripts/analysis/analyze.py",
                    "--results",
                    result_path,
                    "--output-dir",
                    result_path.parent / "notebook_analysis",
                ],
                required_path=result_path,
            )
    else:
        print("Analysis skipped. Set RUN_ANALYSIS = True to analyze", name)
    return paths


def load_results(paths):
    frames = []
    for path in paths:
        frame = pd.read_csv(path, low_memory=False)
        frame["source_file"] = str(path)
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    combined = pd.concat(frames, ignore_index=True)
    combined.attrs["result_paths"] = [str(path) for path in paths]
    return combined


def numeric_mean(series):
    return pd.to_numeric(series, errors="coerce").mean()


def mechanism_snapshot(frame):
    if frame.empty:
        return frame
    group_cols = [col for col in ["experiment_id", "variant_id"] if col in frame.columns]
    if not group_cols:
        group_cols = ["variant_id"] if "variant_id" in frame.columns else []
    rows = []
    grouped = frame.groupby(group_cols, dropna=False) if group_cols else [("all", frame)]
    for key, part in grouped:
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_cols, key))
        row["rows"] = len(part)
        row["mean_payoff"] = numeric_mean(part["payoff"]) if "payoff" in part else None
        row["total_payoff"] = pd.to_numeric(part["payoff"], errors="coerce").sum() if "payoff" in part else None
        row["mean_entropy"] = numeric_mean(part["q_pi_entropy"]) if "q_pi_entropy" in part else None
        row["joint_accuracy"] = numeric_mean(part["inferred_joint_correct"]) if "inferred_joint_correct" in part else None
        row["mean_log_evidence"] = numeric_mean(part["round_log_evidence"]) if "round_log_evidence" in part else None
        row["mean_planning_cost"] = numeric_mean(part["planning_cost_ratio"]) if "planning_cost_ratio" in part else None
        if "selected_partner" in part:
            row["distinct_selected_partners"] = part["selected_partner"].nunique(dropna=True)
        rows.append(row)
    return pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True) if group_cols else pd.DataFrame(rows)


def display_analysis_tables(paths, names=None):
    names = names or [
        "hypothesis_summary.csv",
        "final_round_summary.csv",
        "deployment_dissociation_summary.csv",
        "partner_choice_summary.csv",
        "model_fitness_correlation_summary.csv",
        "betrayal_phase_summary.csv",
        "betrayal_reallocation_summary.csv",
        "phenotype_validation_summary.csv",
    ]
    shown = 0
    for analysis_dir in analysis_dirs_for_paths(paths):
        for name in names:
            table_path = analysis_dir / name
            if table_path.exists():
                print(f"Analysis table: {table_path}")
                display(pd.read_csv(table_path).head(12))
                shown += 1
    if shown == 0:
        print("No compact analysis tables found yet. Run with RUN_ANALYSIS = True to generate them.")


def plot_timecourse(frame, title, value="q_pi_entropy"):
    if frame.empty or value not in frame.columns or "round" not in frame.columns or "variant_id" not in frame.columns:
        print(f"Skipping {value} timecourse; required columns are not available.")
        return None
    curve = frame.groupby(["round", "variant_id"], as_index=False)[value].mean()
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for variant, part in curve.groupby("variant_id"):
        ax.plot(part["round"], part[value], label=str(variant))
    ax.set(title=f"{title}: {value} over rounds", xlabel="Round", ylabel=value)
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    display(fig)
    plt.close(fig)
    return fig


def plot_demo_summary(frame, title):
    if frame.empty:
        print(f"No rows available for {title}.")
        return None
    summary = mechanism_snapshot(frame)
    display(summary)
    metric_cols = [col for col in ["mean_payoff", "mean_entropy", "joint_accuracy"] if col in summary.columns and summary[col].notna().any()]
    if not metric_cols:
        return None
    label_col = "variant_id" if "variant_id" in summary.columns else summary.columns[0]
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(4.6 * len(metric_cols), 3.4), squeeze=False)
    for ax, metric in zip(axes[0], metric_cols):
        ax.bar(summary[label_col].astype(str), summary[metric])
        ax.set(title=metric.replace("_", " "), xlabel="Variant", ylabel=metric)
        ax.tick_params(axis="x", rotation=30)
    fig.suptitle(title)
    plt.tight_layout()
    display(fig)
    plt.close(fig)
    return fig


def plot_profile_metrics(frame, title):
    if frame.empty:
        print(f"No rows available for {title}.")
        return None
    for value in ["q_pi_entropy", "payoff", "round_log_evidence"]:
        if value in frame.columns:
            plot_timecourse(frame, title, value=value)
    if "selected_partner" in frame.columns and "round" in frame.columns:
        selection = frame.groupby(["round", "variant_id"], as_index=False)["selected_partner"].nunique()
        fig, ax = plt.subplots(figsize=(8, 3.4))
        for variant, part in selection.groupby("variant_id"):
            ax.plot(part["round"], part["selected_partner"], label=str(variant))
        ax.set(title=f"{title}: selected-partner spread", xlabel="Round", ylabel="distinct partners")
        ax.legend(loc="best", fontsize=8)
        plt.tight_layout()
        display(fig)
        plt.close(fig)


def interpretation_card(name, frame):
    spec = DEMO_EXPERIMENTS[name]
    print("Question:", spec["question"])
    print("What To Look For:", spec["readout"])
    snapshot = mechanism_snapshot(frame)
    if snapshot.empty:
        print("No data loaded yet.")
        return
    display(snapshot.head(20))
    if "mean_entropy" in snapshot and snapshot["mean_entropy"].notna().any():
        ordered = snapshot.sort_values("mean_entropy")[[col for col in ["experiment_id", "variant_id", "mean_entropy"] if col in snapshot.columns]]
        print("Lower entropy means more decisive policy selection in this demo-scale readout.")
        display(ordered.head(10))
    if "joint_accuracy" in snapshot and snapshot["joint_accuracy"].notna().any():
        ordered = snapshot.sort_values("joint_accuracy", ascending=False)[[col for col in ["experiment_id", "variant_id", "joint_accuracy"] if col in snapshot.columns]]
        print("Joint accuracy is a model-state check, not the whole behavioral story.")
        display(ordered.head(10))


def show_demo(name, frame):
    spec = DEMO_EXPERIMENTS[name]
    if frame.empty:
        print(f"No rows loaded for {spec['label']}.")
        return
    print("Rows:", len(frame))
    plot_demo_summary(frame, spec["label"])
    plot_timecourse(frame, spec["label"], value="q_pi_entropy")


def show_appendix_demo(name, frame):
    spec = DEMO_EXPERIMENTS[name]
    print(f"Appendix-Level Extensions: {spec['label']}")
    show_demo(name, frame)
    plot_profile_metrics(frame, spec["label"])


def run_demo(name):
    spec = DEMO_EXPERIMENTS[name]
    print(f"Demo config: {spec['config']}")
    print(f"Paper analogue: {spec['paper_config']}")
    paths = run_demo_experiment(name)
    frame = load_results(paths)
    if not frame.empty:
        frame.attrs["result_paths"] = [str(path) for path in paths]
    return frame


## 5. Predictability-Value Demo: Run And Analyze

Config: `configs/demo/01_predictability_value.toml`

**Question.** Does partner-local confidence behave like a reliability signal rather than a reward meter?

**What To Look For.** Compare payoff with policy entropy and model-fitness summaries. The interesting pattern is not simply which variant earns more in a tiny run; it is whether precision-related readouts follow predictability and evidence.


In [ ]:
predictability_value_results = run_demo("predictability_value")


In [ ]:
show_demo("predictability_value", predictability_value_results)
interpretation_card("predictability_value", predictability_value_results)
display_analysis_tables([Path(p) for p in predictability_value_results.attrs.get("result_paths", [])], ["model_fitness_correlation_summary.csv", "deployment_dissociation_summary.csv", "final_round_summary.csv"])


## 6. Deployment-Ablation Demo: Run And Analyze

Config: `configs/demo/02_deployment_ablation.toml`

**Question.** Is affective precision doing its work by changing policy deployment, or by giving the model extra knowledge?

**What To Look For.** The tracked-only lesion keeps the confidence tracker visible while blocking the route into action precision. Read entropy and deployment summaries before making a payoff claim.


In [ ]:
deployment_ablation_results = run_demo("deployment_ablation")


In [ ]:
show_demo("deployment_ablation", deployment_ablation_results)
interpretation_card("deployment_ablation", deployment_ablation_results)
display_analysis_tables([Path(p) for p in deployment_ablation_results.attrs.get("result_paths", [])], ["deployment_dissociation_summary.csv", "hypothesis_summary.csv", "final_round_summary.csv"])


## 7. Partner-Selection Demo: Run And Analyze

Config: `configs/demo/03_partner_selection.toml`

**Question.** Once the agent can choose partners, does local confidence change the social sampling pattern?

**What To Look For.** Inspect selected-partner spread and partner-choice summaries. A useful result can be allocation reorganization even when total payoff stays close.


In [ ]:
partner_selection_results = run_demo("partner_selection")


In [ ]:
show_demo("partner_selection", partner_selection_results)
interpretation_card("partner_selection", partner_selection_results)
plot_profile_metrics(partner_selection_results, "Partner selection")
display_analysis_tables([Path(p) for p in partner_selection_results.attrs.get("result_paths", [])], ["partner_choice_summary.csv", "partner_model_fitness_summary.csv", "final_round_summary.csv"])


## 8. Betrayal-Adaptation Demo: Run And Analyze

Config: `configs/demo/04_betrayal_adaptation.toml`

**Question.** What happens when a previously reliable partner changes stance?

**What To Look For.** Read this as a temporal-dependency test. Confidence can sharpen action selection, but after a switch the same sharpness can expose whether the agent reallocates, perseverates, or updates its partner model.


In [ ]:
betrayal_adaptation_results = run_demo("betrayal_adaptation")


In [ ]:
show_demo("betrayal_adaptation", betrayal_adaptation_results)
interpretation_card("betrayal_adaptation", betrayal_adaptation_results)
plot_profile_metrics(betrayal_adaptation_results, "Betrayal adaptation")
display_analysis_tables([Path(p) for p in betrayal_adaptation_results.attrs.get("result_paths", [])], ["betrayal_phase_summary.csv", "betrayal_reallocation_summary.csv", "betrayal_misdeployment_summary.csv", "evidence_effect_summary.csv"])


## 9. Appendix-Level Extensions

The next three demos are included by default. They are useful because the paper's appendix/profile material is where the mechanism becomes less one-dimensional: gain, priors, reengagement, confidence, and payoff can move differently.


## 10. Alpha-Sweep Demo: Run And Analyze

Config: `configs/demo/05a_alpha_sweep.toml`

**Question.** What does the affective gain parameter change?

**What To Look For.** The appendix-level lesson is that gain controls confidence amplitude and policy commitment. A larger confidence movement is not automatically a better payoff regime.


In [ ]:
alpha_sweep_results = run_demo("alpha_sweep")


In [ ]:
show_appendix_demo("alpha_sweep", alpha_sweep_results)
interpretation_card("alpha_sweep", alpha_sweep_results)
display_analysis_tables([Path(p) for p in alpha_sweep_results.attrs.get("result_paths", [])], ["affective_movement_summary.csv", "phenotype_validation_summary.csv", "betrayal_phase_summary.csv", "hypothesis_summary.csv"])


## 11. Prior-Factorial Demo: Run And Analyze

Config: `configs/demo/05b_prior_factorial.toml`

**Question.** How do trust priors and precision gain combine into profile-like behavior?

**What To Look For.** Compare profile rows across open, betrayal, and partner-choice sub-experiments. The point is the shape of the trade-off, not a single winning profile.


In [ ]:
prior_factorial_results = run_demo("prior_factorial")


In [ ]:
show_appendix_demo("prior_factorial", prior_factorial_results)
interpretation_card("prior_factorial", prior_factorial_results)
display_analysis_tables([Path(p) for p in prior_factorial_results.attrs.get("result_paths", [])], ["phenotype_validation_summary.csv", "partner_choice_summary.csv", "betrayal_reallocation_summary.csv", "hypothesis_summary.csv"])


## 12. Forgiveness Demo: Run And Analyze

Config: `configs/demo/05c_forgiveness.toml`

**Question.** Does behavioral reengagement mean confidence has recovered?

**What To Look For.** This demo is intentionally appendix-like. Read reengagement, beta movement, and payoff recovery separately; forgiveness here is a computational dissociation, not a moral label.


In [ ]:
forgiveness_results = run_demo("forgiveness")


In [ ]:
show_appendix_demo("forgiveness", forgiveness_results)
interpretation_card("forgiveness", forgiveness_results)
display_analysis_tables([Path(p) for p in forgiveness_results.attrs.get("result_paths", [])], ["betrayal_reallocation_summary.csv", "betrayal_phase_summary.csv", "affective_movement_summary.csv", "phenotype_validation_summary.csv"])


## 13. What This Demo Does Not Prove

These are small, fast analogues. They prove that the public configs, runner, analysis scripts, and visual readouts work together. They do not replace the paper-scale result cards under `results/paper/` or the manuscript source tables.

Use `notebooks/reproduce.ipynb` when the goal is full paper reproduction. Use this notebook when the goal is to understand the mechanism before waiting for larger runs.
